In [0]:
# Connect to ADLS
storage_account_name = "azureetlstorage"

storage_account_key = dbutils.secrets.get(
    scope = "etl-scope",
    key   = "adls-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

print("connected!")

In [0]:
# Read Bronze reviews
reviews_bronze_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/bronze/reviews"

bronze_reviews_df = spark.read.format("delta").load(reviews_bronze_path)

print(f"Bronze reviews: {bronze_reviews_df.count()}")
bronze_reviews_df.printSchema()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# Step 1 — Fix data types and clean
typed_df = bronze_reviews_df \
    .withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd")) \
    .withColumn("reviewer", F.trim(F.col("reviewer"))) \
    .withColumn("text", F.trim(F.col("text")))

# Step 2 — Data quality checks
validated_df = typed_df \
    .withColumn(
        "_dq_issues",
        F.array_except(
            F.array(
                F.when(F.col("review_id").isNull(),  F.lit("NULL_REVIEW_ID")).otherwise(F.lit(None)),
                F.when(F.col("product_id").isNull(), F.lit("NULL_PRODUCT_ID")).otherwise(F.lit(None)),
                F.when(F.col("rating").isNull(),     F.lit("NULL_RATING")).otherwise(F.lit(None)),
                F.when(F.col("rating") < 1,          F.lit("INVALID_RATING")).otherwise(F.lit(None)),
                F.when(F.col("rating") > 5,          F.lit("INVALID_RATING")).otherwise(F.lit(None)),
            ),
            F.array(F.lit(None).cast("string"))
        )
    ) \
    .withColumn("_is_valid", F.size(F.col("_dq_issues")) == 0)

# Show results
invalid = validated_df.where(F.col("_is_valid") == False)
print(f"Invalid reviews: {invalid.count()}")
print(f"Valid reviews:   {validated_df.where(F.col('_is_valid') == True).count()}")
invalid.show(5)

In [0]:
# Step 3 — Separate clean and quarantine
clean_df = validated_df.where(F.col("_is_valid") == True)
quarantine_df = validated_df.where(F.col("_is_valid") == False)

# Save quarantine
quarantine_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/bronze/reviews_quarantine"
quarantine_df.write.format("delta").mode("overwrite").save(quarantine_path)
print(f"Quarantine saved: {quarantine_df.count()} rows")

# Step 4 — Deduplicate by review_id
window = Window.partitionBy("review_id").orderBy(F.desc("_ingested_at"))

silver_reviews_df = clean_df \
    .withColumn("_row_num", F.row_number().over(window)) \
    .where(F.col("_row_num") == 1) \
    .drop("_row_num") \
    .withColumn("_silver_processed_at", F.current_timestamp())

print(f"Silver reviews: {silver_reviews_df.count()} rows")

In [0]:
# Write to Silver
silver_reviews_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/silver/reviews"

silver_reviews_df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("product_id") \
    .save(silver_reviews_path)

print(f"Silver reviews written — {silver_reviews_df.count()} rows")